# Funding Arbitrage Research

Analyse `logs/funding_snapshots.csv`,
`logs/funding_opportunities.csv`, and `logs/funding_positions.csv`
produced by either :

- `python scripts/scan_funding_opportunities.py ...`, or
- `python engine_v9.py --paper --config config/presets/paper_500_funding_research.json`.

**Paper / research only.** No live orders are routed.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
REPO_ROOT = Path('.').resolve()
if (REPO_ROOT / 'logs').exists() == False:
    REPO_ROOT = REPO_ROOT.parent
SNAP = REPO_ROOT / 'logs' / 'funding_snapshots.csv'
OPP  = REPO_ROOT / 'logs' / 'funding_opportunities.csv'
POS  = REPO_ROOT / 'logs' / 'funding_positions.csv'
for p in (SNAP, OPP, POS):
    print(p, 'exists:', p.exists())

## 1. Load funding data

In [ ]:
snaps = pd.read_csv(SNAP) if SNAP.exists() else pd.DataFrame()
opps  = pd.read_csv(OPP) if OPP.exists() else pd.DataFrame()
positions = pd.read_csv(POS) if POS.exists() else pd.DataFrame()
print('snaps:', len(snaps), '| opps:', len(opps), '| positions:', len(positions))
snaps.head()

## 2. Funding distribution per exchange × symbol (hourly bps)

In [ ]:
if not snaps.empty:
    snaps['funding_rate_bps'] = pd.to_numeric(snaps.get('funding_rate_bps'), errors='coerce')
    snaps.groupby(['exchange', 'symbol'])['funding_rate_bps'].describe()

## 3. Funding persistence

Corr(F_t, F_{t+1}), Corr(F_t, F_{t+4}), Corr(F_t, F_{t+8}) per exchange × symbol.

In [ ]:
def lag_corr(s, lag):
    s = pd.to_numeric(s, errors='coerce').dropna()
    return float(s.corr(s.shift(lag))) if len(s) > lag + 5 else float('nan')

rows = []
if not snaps.empty:
    for (exch, sym), g in snaps.sort_values('ts').groupby(['exchange', 'symbol']):
        rows.append({
            'exchange': exch, 'symbol': sym,
            'n': len(g),
            'auto_lag_1': lag_corr(g['funding_rate_bps'], 1),
            'auto_lag_4': lag_corr(g['funding_rate_bps'], 4),
            'auto_lag_8': lag_corr(g['funding_rate_bps'], 8),
        })
pd.DataFrame(rows)

## 4. Funding regime

In [ ]:
def regime(s):
    if abs(s.mean()) > 3 and s.std() / max(abs(s.mean()), 1e-9) < 0.5:
        return 'persistent_high'
    if abs(s.mean()) < 1:
        return 'mean_reverting' if s.std() > 1 else 'flat'
    return 'unstable'

if not snaps.empty:
    snaps.groupby(['exchange', 'symbol'])['funding_rate_bps'].apply(regime)

## 5. Expected carry vs realized carry

Predicted by snapshot at t : `funding_rate_bps * horizon_h`. Realized by
averaging the snapshots over the horizon.

In [ ]:
if not snaps.empty:
    s = snaps.sort_values('ts').copy()
    s['next_8h_avg'] = (s.groupby(['exchange','symbol'])['funding_rate_bps']
                         .transform(lambda x: x.rolling(8, min_periods=1).mean().shift(-8)))
    s[['exchange','symbol','funding_rate_bps','next_8h_avg']].tail()

## 6. Funding spread Hyperliquid ↔ Aster

If only Hyperliquid has data (Aster adapter not wired yet), this
section is empty — that's expected.

In [ ]:
if not snaps.empty:
    pivot = snaps.pivot_table(index=['symbol','ts'], columns='exchange',
                              values='funding_rate_bps', aggfunc='first').reset_index()
    if {'hyperliquid','aster'}.issubset(pivot.columns):
        pivot['spread_bps'] = pivot['hyperliquid'] - pivot['aster']
        pivot[['symbol','ts','hyperliquid','aster','spread_bps']].dropna().head(20)
    else:
        print('Aster snapshots missing — cross-exchange section skipped.')

## 7. Net carry after costs

Opportunity logger already records `expected_net_bps` / `expected_net_usd`. Aggregate by decision.

In [ ]:
if not opps.empty:
    cols = [c for c in ('decision','mode','direction','expected_net_bps','expected_net_usd','reason')
            if c in opps.columns]
    opps[cols].groupby(['decision','mode','direction']).agg(['mean','median','count']).round(4)

## 8. Basis & 9. Liquidation buffer — placeholders

Basis is included in `funding_opportunities.csv` (`basis_bps`).
Liquidation buffer requires margin data not yet stored in this
framework — TODO when Aster live execution is wired.

## 10. Paper backtest (cross-exchange)

Approximation : for each `PAPER_CROSS_EXCHANGE_CANDIDATE`, simulate a
1-hour hold and check whether the next-window realized funding spread
covered costs.

In [ ]:
if not opps.empty:
    candidates = opps[opps.get('decision').astype(str).str.contains('PAPER_CROSS_EXCHANGE_CANDIDATE', na=False)] \
        if 'decision' in opps.columns else pd.DataFrame()
    candidates.head(20)

## 11. Ranking — best symbols by stable net bps

In [ ]:
if not opps.empty and 'symbol' in opps.columns and 'expected_net_bps' in opps.columns:
    opps.groupby('symbol')['expected_net_bps'].agg(['mean','median','std','count']).sort_values('median', ascending=False)

## 12. Export Markdown report

In [ ]:
report_lines = [
    '# Funding Arbitrage Research Report',
    f'- snapshots rows: {len(snaps)}',
    f'- opportunities rows: {len(opps)}',
    f'- positions rows: {len(positions)}',
    '',
    'See notebook for details. Reminder of validation gates :',
    '- funding elevated AND persistent,',
    '- funding spread between exchanges stable,',
    '- net carry positive after costs,',
    '- basis risk low,',
    '- liquidity sufficient,',
    '- no abrupt funding reversal,',
    '- hedge error tiny,',
    '- liquidation buffer large.',
]
out_path = REPO_ROOT / 'reports' / 'funding_arbitrage_report.md'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text('\n'.join(report_lines), encoding='utf-8')
print('Wrote', out_path)